# Phase 2 — สร้างชุดข้อมูลโดเมน (Q&A สำนักงานการทะเบียน จุฬาฯ)

**เป้าหมาย:** ได้ชุด Q&A ไทย-อังกฤษ คุณภาพสูง ~2,000+ คู่ ใช้ซ้ำ 3 ที่: calibration (Phase 4), recovery training (Phase 5), evaluation (Phase 1/7)

> ⚠️ **ขั้นเสี่ยงสูงสุดของทั้งโปรเจกต์** — ดาตาน้อย/คุณภาพต่ำ = พังทั้งงาน คุณภาพ > ปริมาณ

---
### 🔴 หมายเหตุสำคัญ — รันจริงทำที่ `scripts/` (local) ไม่ใช่ notebook นี้

โปรเจกต์นี้ทำ Phase 2 แบบ **local CPU** (ไม่ใช่ Kaggle/PDF) เพราะ:
- **โดเมนจริง = สำนักงานการทะเบียน จุฬาฯ** ตอบทุกอย่างใน https://www.reg.chula.ac.th/th/ (ใช้คำว่า "นิสิต") — ยังไม่มีเอกสาร PDF จึง **crawl เว็บ** เป็นแหล่ง
- LLM generator = **Gemini `gemini-2.5-flash`** (key ใน `.env` → `GEMINI_API_KEY`)

ไปป์ไลน์จริง (รันตามลำดับ):
```powershell
# ใน .tools\activate.ps1 ก่อน แล้ว:
$env:PYTHONUTF8="1"
.venv\Scripts\python.exe scripts\phase2_scrape.py   --max-pages 150 --max-pdf 40   # → data/chunks.jsonl, corpus_raw.txt
.venv\Scripts\python.exe scripts\phase2_generate.py --pairs 6                       # → data/qa_raw.jsonl  (resumable)
.venv\Scripts\python.exe scripts\phase2_split.py                                    # → data/{train,val,test}.jsonl + dataset_card.md
```
env ต้องมี: `truststore` (แก้ proxy MITM ของเครือข่าย) + `PYTHONUTF8=1` (กัน console cp874 crash ตอน print ไทย)

**Output → push เป็น Kaggle Dataset (ตอนจะรัน Phase 3+ บน Kaggle):** `train.jsonl`, `val.jsonl`, `test.jsonl`, `corpus_raw.txt`, `dataset_card.md`

> เซลล์ด้านล่างเป็นเวอร์ชัน Kaggle/PDF เดิม เก็บไว้อ้างอิง — ตรรกะหลักย้ายไป `scripts/phase2_*.py` แล้ว

In [ ]:
%pip install -q datasets pdfplumber google-generativeai
import os, json, glob, random, hashlib, time, re

INPUT_DIR = "/kaggle/input"   # attach แหล่งเอกสารดิบ (PDF) ที่นี่
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
random.seed(42)
print("inputs:", os.listdir(INPUT_DIR) if os.path.isdir(INPUT_DIR) else "(none)")

## 1. อ่าน PDF → chunk
วาง PDF (ระเบียบ/FAQ/คู่มือ) เป็น Kaggle Dataset แล้ว attach • โค้ดสแกน `*.pdf` ทุกตัวใน `/kaggle/input` แล้ว extract text + แบ่ง chunk
> ถ้ายังไม่มี PDF จะใช้ `SAMPLE_DOCS` เพื่อทดสอบ pipeline ทั้งสายให้รันผ่านก่อน

In [ ]:
import pdfplumber

CHUNK_CHARS = 1500       # ขนาด chunk โดยประมาณ (ตัวอักษร)
CHUNK_OVERLAP = 200      # overlap กันความรู้ขาดช่วงรอยต่อ

def chunk_text(text, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i + size])
        i += size - overlap
    return [c for c in out if c.strip()]

# ข้อความตัวอย่าง (fallback) — ใช้ทดสอบ pipeline เมื่อยังไม่มี PDF จริง
SAMPLE_DOCS = [
    {"source": "sample", "text": (
        "การขอทุนการศึกษา: นักศึกษายื่นคำร้องผ่านระบบออนไลน์ของกองกิจการนักศึกษา "
        "ภายในวันที่ 30 มิถุนายนของทุกปี เอกสารที่ต้องใช้ ได้แก่ สำเนาบัตรนักศึกษา "
        "ใบแสดงผลการเรียน และหนังสือรับรองรายได้ครอบครัว "
        "หอพักนักศึกษา: เปิดลงทะเบียนเข้าพักช่วงเดือนพฤษภาคม ก่อนเปิดภาคเรียนที่ 1 "
        "ค่าหอพักชำระเป็นรายภาคการศึกษา นักศึกษาชั้นปีที่ 1 ได้รับสิทธิ์ก่อน")},
]

docs = []
for fp in glob.glob(f"{INPUT_DIR}/**/*.pdf", recursive=True):
    try:
        with pdfplumber.open(fp) as pdf:
            full = "\n".join((pg.extract_text() or "") for pg in pdf.pages)
        for ck in chunk_text(full):
            docs.append({"source": os.path.basename(fp), "text": ck})
    except Exception as e:
        print("ข้าม", fp, "->", e)

if not docs:
    print("⚠️ ไม่พบ PDF — ใช้ SAMPLE_DOCS เพื่อทดสอบ pipeline")
    docs = [{"source": d["source"], "text": ck}
            for d in SAMPLE_DOCS for ck in chunk_text(d["text"])]

print("chunks:", len(docs), "| sources:", sorted({d['source'] for d in docs}))

## 2. สร้าง Q&A สังเคราะห์ด้วย Gemini API
ตั้ง `GOOGLE_API_KEY` ใน **Add-ons → Secrets** • prompt บังคับให้คำตอบอิงเอกสารเท่านั้น + หลายสไตล์ (ทางการ/ไม่ทางการ, ไทย/อังกฤษ/ปนกัน)

In [ ]:
import google.generativeai as genai

# --- ตั้งค่า API key (Kaggle Secrets: GOOGLE_API_KEY) ---
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except Exception:
        pass
genai.configure(api_key=api_key)

GEN_MODEL = "gemini-1.5-flash"     # flash = เร็ว/ฟรี-tier เยอะ; เปลี่ยนเป็น -pro ถ้าต้องการคุณภาพสูงขึ้น
PAIRS_PER_CHUNK = 5
MAX_CHUNKS = None                  # จำกัดจำนวน chunk ตอนทดสอบ (None = ทั้งหมด)

GEN_PROMPT = '''จากเอกสารด้านล่าง สร้างคู่ถาม-ตอบเกี่ยวกับกิจการนักศึกษา {n} คู่
- หลากหลายสไตล์: ทางการ/ไม่ทางการ, ไทย/อังกฤษ/ปนกัน
- คำตอบต้องอิงข้อเท็จจริงจากเอกสารเท่านั้น ห้ามแต่งข้อมูลที่ไม่มีในเอกสาร
- ตอบเป็น JSON array เท่านั้น ไม่ต้องมีข้อความอื่น:
[{{"question": "...", "answer": "...", "lang": "th|en|mix", "style": "formal|casual"}}]

เอกสาร:
{context}'''

def _parse_json_array(text):
    # ดึง JSON array ออกจาก response (กัน markdown fence / ข้อความเกิน)
    m = re.search(r"\[.*\]", text, re.DOTALL)
    if not m:
        return []
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return []

def generate_qa(context, n=PAIRS_PER_CHUNK, retries=3):
    model = genai.GenerativeModel(GEN_MODEL)
    prompt = GEN_PROMPT.format(n=n, context=context)
    for attempt in range(retries):
        try:
            resp = model.generate_content(prompt)
            rows = _parse_json_array(resp.text)
            return [r for r in rows if r.get("question") and r.get("answer")]
        except Exception as e:
            if attempt == retries - 1:
                print("  generate ล้มเหลว:", e)
                return []
            time.sleep(2 * (attempt + 1))   # backoff กัน rate limit

qa_pairs = []
chunks = docs if MAX_CHUNKS is None else docs[:MAX_CHUNKS]
for i, d in enumerate(chunks):
    rows = generate_qa(d["text"])
    for r in rows:
        r["source"] = d["source"]
    qa_pairs.extend(rows)
    if (i + 1) % 10 == 0 or i == len(chunks) - 1:
        print(f"  {i+1}/{len(chunks)} chunks → {len(qa_pairs)} คู่")

print("generated:", len(qa_pairs))

## 3. ดักซ้ำ (dedup) เบื้องต้น — ก่อนคัดด้วยมือ

In [ ]:
def _norm(q):
    return "".join(q.split()).lower()

seen, deduped = set(), []
for ex in qa_pairs:
    h = hashlib.md5(_norm(ex["question"]).encode()).hexdigest()
    if h not in seen:
        seen.add(h); deduped.append(ex)
print(f"{len(qa_pairs)} → {len(deduped)} หลัง dedup")

# เซฟไว้ให้คัดด้วยมือ (เปิดไฟล์นี้ใน editor แล้วลบ/แก้ที่ไม่ดี)
raw_path = os.path.join(OUT_DIR, "qa_for_review.jsonl")
with open(raw_path, "w", encoding="utf-8") as f:
    for ex in deduped:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print("เซฟเพื่อคัดด้วยมือ:", raw_path)

## 4. ✋ คัดกรองด้วยมือ (นอก notebook)
ตัดคำตอบผิด/กำกวม/ซ้ำ → อัป `qa_clean.jsonl` กลับมาเป็น Kaggle Dataset แล้ว attach

In [ ]:
# โหลดเวอร์ชันที่คัดมือแล้ว (ถ้ายังไม่มี ใช้ deduped ไปก่อนเพื่อทดสอบ pipeline)
clean_fp = f"{INPUT_DIR}/<dataset>/qa_clean.jsonl"
if os.path.exists(clean_fp):
    clean = [json.loads(l) for l in open(clean_fp, encoding="utf-8")]
else:
    clean = deduped
print("clean:", len(clean))

## 5. แบ่ง train/val/test (80/10/10) — ห้าม test รั่ว!

In [ ]:
random.shuffle(clean)
n = len(clean)
n_train, n_val = int(n * 0.8), int(n * 0.1)
splits = {
    "train": clean[:n_train],
    "val": clean[n_train:n_train + n_val],
    "test": clean[n_train + n_val:],
}

for name, rows in splits.items():
    fp = os.path.join(OUT_DIR, f"{name}.jsonl")
    with open(fp, "w", encoding="utf-8") as f:
        for ex in rows:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"{name}: {len(rows)} → {fp}")

# ✅ ตรวจไม่มี test รั่วไป train/val
test_q = {_norm(e["question"]) for e in splits["test"]}
leak = test_q & {_norm(e["question"]) for e in splits["train"] + splits["val"]}
assert not leak, f"❌ พบ test รั่ว {len(leak)} ข้อ!"
print("✅ ไม่มี test รั่ว")

## 6. corpus ดิบ (ไทย-อังกฤษล้วน) สำหรับ vocab analysis (Phase 3)

In [ ]:
corpus_fp = os.path.join(OUT_DIR, "corpus_raw.txt")
with open(corpus_fp, "w", encoding="utf-8") as f:
    for d in docs:
        f.write(d["text"] + "\n")
    for ex in clean:           # รวมข้อความ Q&A เข้า corpus ด้วย
        f.write(ex["question"] + "\n" + ex["answer"] + "\n")
print("corpus:", corpus_fp)

## 7. dataset card + push

**ขั้นต่อไป:** Save Version แล้วสร้าง/อัปเดต Kaggle Dataset จาก output (`train/val/test.jsonl`, `corpus_raw.txt`) เพื่อให้ Phase 3–5 attach ได้

In [ ]:
card = f'''# Dataset: กิจการนักศึกษา Q&A (v1)
- total: {len(clean)} | train/val/test: {len(splits["train"])}/{len(splits["val"])}/{len(splits["test"])}
- วิธีสร้าง: synthetic จากเอกสารจริง + คัดด้วยมือ
- การใช้งาน: calibration (P4), recovery (P5), eval (P1/P7)
- ⚠️ test แยกเด็ดขาด ห้ามใช้ใน calibration/training
'''
open(os.path.join(OUT_DIR, "dataset_card.md"), "w", encoding="utf-8").write(card)
print(card)